# 03 - Diagnostics, Regularization, and Interpretability

This notebook goes beyond model fitting. It teaches how to inspect assumptions, identify multicollinearity, compare OLS with Ridge/Lasso, and write defensible conclusions.


## Learning objectives

Students will learn how to:

1. Inspect residual behavior.
2. Run Breusch-Pagan and Jarque-Bera diagnostics.
3. Calculate VIF for multicollinearity.
4. Compare Linear Regression, Ridge, and Lasso using cross-validation.
5. Interpret regularization as coefficient shrinkage.


## 1. Import libraries


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

DATA_DIR = Path.cwd().parent / "data" if (Path.cwd().parent / "data").exists() else Path.cwd() / "data"


## 2. Load data


In [ ]:
df = pd.read_csv(DATA_DIR / "multiple_regression_marketing_sales.csv")
df.head()


## 3. Fit statsmodels OLS


In [ ]:
ols = smf.ols(
    "sales_k_units ~ tv_spend_k + radio_spend_k + social_spend_k + price_index + competitor_spend_k + C(season)",
    data=df,
).fit()

print(f"R2          : {ols.rsquared:.4f}")
print(f"Adjusted R2: {ols.rsquared_adj:.4f}")
print(f"AIC         : {ols.aic:.2f}")
print(f"BIC         : {ols.bic:.2f}")


## 4. Residual plots


In [ ]:
residuals = ols.resid
fitted = ols.fittedvalues

plt.figure(figsize=(7, 5))
plt.scatter(fitted, residuals, alpha=0.8)
plt.axhline(0, linewidth=1)
plt.title("OLS residuals vs fitted values")
plt.xlabel("Fitted values")
plt.ylabel("Residuals")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))
plt.hist(residuals, bins=15, edgecolor="black")
plt.title("OLS residual distribution")
plt.xlabel("Residual")
plt.ylabel("Count")
plt.grid(True, alpha=0.3)
plt.show()


## 5. Diagnostic tests

| Test | Purpose |
|---|---|
| Breusch-Pagan | Checks heteroscedasticity |
| Jarque-Bera | Checks residual normality |


In [ ]:
bp_stat, bp_p_value, _, _ = het_breuschpagan(residuals, ols.model.exog)
jb_result = stats.jarque_bera(residuals)

diagnostics = pd.DataFrame({
    "diagnostic": ["R2", "Adjusted R2", "Breusch-Pagan p-value", "Jarque-Bera p-value", "AIC", "BIC"],
    "value": [ols.rsquared, ols.rsquared_adj, bp_p_value, jb_result.pvalue, ols.aic, ols.bic],
})
diagnostics


## 6. Multicollinearity with VIF


In [ ]:
numeric_features = ["tv_spend_k", "radio_spend_k", "social_spend_k", "price_index", "competitor_spend_k"]

X_vif = pd.get_dummies(df[numeric_features + ["season"]], drop_first=True)
X_vif = sm.add_constant(X_vif).astype(float)

vif_table = pd.DataFrame({
    "feature": X_vif.columns,
    "vif": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])],
}).sort_values("vif", ascending=False)

vif_table


## 7. Why regularization?

OLS can become unstable when predictors are noisy, numerous, or correlated.

| Model | Penalty | Main use |
|---|---|---|
| Ridge | L2 | Shrinks coefficients, keeps all variables |
| Lasso | L1 | Shrinks coefficients and can select variables |


## 8. Build preprocessing pipeline for regularized models


In [ ]:
target = "sales_k_units"
categorical_features = ["season"]

X = df[numeric_features + categorical_features]
y = df[target]

preprocess_scaled = ColumnTransformer([
    ("categorical", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features),
    ("numeric", StandardScaler(), numeric_features),
])


## 9. Compare Linear Regression, Ridge, and Lasso


In [ ]:
models = {
    "LinearRegression": LinearRegression(),
    "Ridge_alpha_1": Ridge(alpha=1.0),
    "Ridge_alpha_10": Ridge(alpha=10.0),
    "Lasso_alpha_0_05": Lasso(alpha=0.05, random_state=43, max_iter=10000),
    "Lasso_alpha_0_5": Lasso(alpha=0.5, random_state=43, max_iter=10000),
}

cv = KFold(n_splits=5, shuffle=True, random_state=43)
rows = []

for name, estimator in models.items():
    pipeline = Pipeline([
        ("preprocess", preprocess_scaled),
        ("model", estimator),
    ])
    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring={"r2": "r2", "mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error"},
    )
    rows.append({
        "model": name,
        "mean_cv_r2": scores["test_r2"].mean(),
        "mean_cv_mae": -scores["test_mae"].mean(),
        "mean_cv_rmse": -scores["test_rmse"].mean(),
    })

comparison = pd.DataFrame(rows).sort_values("mean_cv_rmse")
comparison


## 10. Visualize model comparison


In [ ]:
plot_data = comparison.sort_values("mean_cv_rmse", ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(plot_data["model"], plot_data["mean_cv_rmse"])
plt.title("Model comparison by cross-validated RMSE")
plt.xlabel("Mean CV RMSE")
plt.ylabel("Model")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


## 11. Coefficient shrinkage demonstration


In [ ]:
feature_names = None
coef_rows = []

for name in ["LinearRegression", "Ridge_alpha_10", "Lasso_alpha_0_5"]:
    estimator = models[name]
    pipeline = Pipeline([
        ("preprocess", preprocess_scaled),
        ("model", estimator),
    ])
    pipeline.fit(X, y)
    feature_names = pipeline.named_steps["preprocess"].get_feature_names_out()
    coefficients = pipeline.named_steps["model"].coef_
    for feature, coefficient in zip(feature_names, coefficients):
        coef_rows.append({
            "model": name,
            "feature": feature,
            "coefficient": coefficient,
        })

coef_compare = pd.DataFrame(coef_rows)
coef_compare.head(15)


## 12. Visualize coefficient shrinkage


In [ ]:
pivot = coef_compare.pivot(index="feature", columns="model", values="coefficient")

pivot.plot(kind="barh", figsize=(10, 7))
plt.axvline(0, linewidth=1)
plt.title("Coefficient comparison: OLS vs Ridge vs Lasso")
plt.xlabel("Coefficient value")
plt.ylabel("Feature")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


## 13. Final diagnostic interpretation

A good regression conclusion should include:

1. Predictive performance.
2. Residual behavior.
3. Heteroscedasticity check.
4. Residual normality check.
5. Multicollinearity check.
6. Regularization comparison.
7. Practical recommendation.


## Student exercises

1. Increase Ridge alpha and observe coefficient shrinkage.
2. Increase Lasso alpha and identify coefficients that become zero.
3. Add another diagnostic plot.
4. Remove `season` and rerun VIF.
5. Write a final model governance note: prediction use, interpretation risk, and limitations.
